# Impedance ML Engineering — Session 15
## Calibrate probabilities and choose a decision rule
**Core: 60 minutes · Classification · Optional extension: 20–40 minutes**

## Why this matters
Are predicted probabilities meaningful, and how does a chosen error cost affect the operating threshold?

Build the judgment to implement, review, and challenge data and modeling workflows in an ML engineering role.

## Learning objectives
- Generate patient-grouped out-of-fold probabilities.
- Separate model fit, calibration, threshold choice, and test.
- Inspect reliability, Brier score, and precision-recall tradeoffs.
- Choose a threshold using an explicit hypothetical loss.

## R-to-Python / prerequisite bridge
Out-of-fold predictions require each row to be predicted without its patient in training; decision threshold differs from the 7.0 kΩ outcome-definition cutoff.

## Suggested pacing
Tasks 1–3: 18 min · Tasks 4–6: 23 min · Tasks 7–8: 11 min · Retrieval/reasoning/reflection: 8 min. Task timings are approximate; if a foundational operation needs repetition, pause there and continue later. Extensions are outside the core hour.

## Inputs and analysis contract
Start Jupyter from the repository root and select **Python (learning-python)**. Read `data/ci_telemetry/README.md` for the complete dictionary. All inputs are local, fictional teaching data: 96 patients, one implant each, 360 visits, 144 directed cells per visit. Matrix rows stimulate; columns record; contact indices are 1–12. `voltage_mv / stim_current_ua` is a synthetic kΩ V/I proxy, not a confirmed manufacturer clinical impedance algorithm. QC-valid means `ok` AND finite voltage. Mask invalid cells; unknown does not mean measured zero. No hearing, disease, or device-failure labels are available.

Write explanations in new markdown cells or print them. Exercise cells are intentionally empty. No downloads or GPU are required.

These later lessons are independently runnable: input scaffolding supplies raw representations and fixed patient indices, not model fits or engineered features. Put this setup in your first exercise cell, then add the requested code:

```python
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from agent.course_data import load_problem, patient_partitions, NUMERIC, CATEGORICAL
problem = load_problem()
part = patient_partitions(problem)
pairs = problem["pairs"]
X = problem["X"]
y = problem["y"]
groups = problem["groups"]
matrices = problem["matrices"]   # raw current values, NaN where invalid
mask = problem["mask"]          # True where observed and usable
train, val, test = part["train"], part["val"], part["test"]
```

`train`, `val`, `test` are positions in the full prediction table. Patient counts are 54/18/24; pair counts 146/51/67. Keep IDs outside features. `target_*` fields and exact future timing are audit/outcome fields. Learn preprocessing only from the applicable training portion. Test outcomes stay reserved until Session 20. For a development-CV lesson, combine train+val, then fit all transformations inside each fold. Earlier validation is repeatedly used for learning; it is exploratory, not independent evidence.

## Concept and API reminders
Calibration asks whether cases assigned probability p occur about p of the time. It differs from discrimination (ranking) and from a decision threshold (an action rule). A calibration map is another fitted model, so it needs predictions produced out of sample for the base learner.

Platt-style calibration fits a logistic map from base logits to outcomes. Patient-grouped OOF logits keep each patient outside the corresponding base fit. The frozen map is then applied to logits from the final training fit. Small calibration samples can add variance and harm probability quality.

For costs c_FN and c_FP, empirical decision cost is `(c_FN×FN + c_FP×FP)/N`; the assumed costs are hypothetical here. Imports: `calibration_curve` from `sklearn.calibration`, `log_loss`, `precision_recall_curve` from `sklearn.metrics`.

## Task 1 — Restate the endpoint and stages (5 min)
Use the Session 14 fixed binary endpoint. Keep 54 training patients, 18 validation patients, 24 test patients. Use training grouped OOF predictions to fit calibration; use validation to inspect calibration and choose a decision threshold. Predict why calibrating on a classifier’s own fitted training predictions is optimistic.

## Task 2 — Generate grouped OOF logits (10 min)
Within training use GroupKFold(3), fitting a fresh logistic pipeline per fold. Collect `.decision_function` values at the original held-out row positions; ensure each row gets exactly one value. Assert patient disjointness. Keep default LogisticRegression C=1 and max_iter 1000 so selection is not nested into this exercise.

## Task 3 — Fit a simple calibration map (7 min)
Fit a second LogisticRegression(C=1e6,max_iter=1000) from one-column OOF logits to training labels (Platt-style mapping). Refit the base pipeline on all training rows. Apply the map to validation base logits. Explain that the mapping is learned and must be saved with the base model; it is not a causal correction.

## Task 4 — Compare probability quality (7 min)
Compare raw and calibrated validation Brier score/log loss plus 5-bin reliability plots with bin counts. Use `calibration_curve` and display counts separately. Explain why calibration on this small sample can worsen results and why empty bins are not evidence of reliability.

## Task 5 — Define a hypothetical decision cost (5 min)
For practice only, set false-negative cost 3 and false-positive cost 1. Compare validation expected cost per row at thresholds 0.1 through 0.9. Keep the clinical meaning absent: this is an operational arithmetic exercise. Distinguish the probability decision threshold from the fixed outcome cutoff.

## Task 6 — Choose and record a threshold (7 min)
Choose one threshold minimizing the prespecified validation cost, with ties broken toward higher threshold. Report confusion matrix, precision, recall, and patient/row counts. Plot precision-recall tradeoff. Do not tune on test; repeatedly selecting from validation can overfit.

## Task 7 — Assess uncertainty and prevalence sensitivity (7 min)
Resample validation patients with replacement 200 times, retaining all pairs per draw, to inspect variability of cost at the FROZEN chosen threshold. Do not reoptimize threshold inside this fixed-rule uncertainty exercise. Explain how changed deployment prevalence can alter precision even at the same sensitivity/specificity.

## Task 8 — Write the decision artifact (4 min)
Save outcome definition, classifier parameters, calibration map, probability threshold, cost assumption, and validation results under `artifacts/15/`. State that independent evaluation is still required and operational costs need domain input.

## Retrieval practice
Explain how to detect a grouped OOF array with missing rows or two predictions for the same row.

## Professional reasoning
1. What could go wrong if calibration folds split rows rather than patients?
2. Why does threshold optimization differ from probability calibration?
3. Why might isotonic calibration be unstable with few independent patients?

## Self-check and reflection
Record confidence **1–5**, hardest point, one thing now explainable, what needs repetition, and workload (too light / about right / too heavy). Restart the kernel and run your attempted cells in order. Ask for review when ready; generation does not count as completion.

## Optional extension — outside the core hour
Try grouped calibration inside an outer grouped evaluation loop on development data; explain which data are allowed to fit the base learner, calibration map, and threshold at each stage.

## API references
[scikit-learn grouped cross-validation](https://scikit-learn.org/stable/modules/cross_validation.html#cross-validation-iterators-for-grouped-data)